# Notebook 04: Offline Reinforcement Learning (Discrete CQL) Training
Prepares Offline RL transitions (state, action, reward, next_state, done), constructs PyTorch datasets with zero patient leakage, and trains the Conservative Q-Learning agent.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import config
from rl.state import StateConstructor
from rl.transitions import generate_offline_transitions
from rl.dataset import create_rl_dataloaders
from rl.model import DiscreteCQLNetwork
from rl.train import train_cql_agent

## 1. Load Processed Trajectory Dataset

In [ ]:
traj_path = config.PROCESSED_DATA_DIR / "trajectory_data.csv"
if not traj_path.exists():
    from main import run_demo_pipeline
    print("Running initial pipeline to build trajectory dataset...")
traj_df = pd.read_csv(traj_path)
print(f"Loaded {len(traj_df)} trajectory records.")
traj_df.head(5)

## 2. Inspect Offline RL State Representation and Actions

In [ ]:
sc = StateConstructor()
print(f"State dimensionality: {sc.state_dim}")
print("State features:", sc.feature_names[:10], "... [total", len(sc.feature_names), "]")
print("Action space:", config.ACTION_NAMES)

## 3. Create Offline RL DataLoaders & Train Discrete CQL Agent

In [ ]:
train_loader, val_loader, train_ds, val_ds = create_rl_dataloaders(
    trajectory_df=traj_df,
    state_constructor=sc,
    batch_size=64,
    val_split=0.2,
)

model, history = train_cql_agent(
    train_loader=train_loader,
    val_loader=val_loader,
    state_dim=sc.state_dim,
    epochs=25,
    lr=3e-4,
    cql_alpha=1.0,
)

## 4. Plot Training Convergence & Loss Curves

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(16, 4))

axs[0].plot(history["epoch"], history["td_loss"], label="Train TD Loss", color="#0d6efd")
axs[0].plot(history["epoch"], history["val_td_loss"], label="Val TD Loss", color="#dc3545", linestyle="--")
axs[0].set_title("Bellman Temporal Difference (TD) Loss")
axs[0].set_xlabel("Epoch")
axs[0].legend()

axs[1].plot(history["epoch"], history["cql_loss"], label="CQL Penalty", color="#ffc107")
axs[1].set_title("Conservative Q-Learning (CQL) Penalty")
axs[1].set_xlabel("Epoch")
axs[1].legend()

axs[2].plot(history["epoch"], history["avg_q"], label="Avg Q-value", color="#198754")
axs[2].set_title("Average Policy Q-Value")
axs[2].set_xlabel("Epoch")
axs[2].legend()

plt.tight_layout()
plt.show()